# Notebook 11 — Análisis de Residuos · Modelo Baseline (RF v1)

**Curso:** DS3022 — Desarrollo de Productos de Datos.
**Producto:** ubIcA (precio de referencia de alquiler en Lima).
**Modelo analizado:** Random Forest v1 (74 features).
**Modelo en producción:** XGBoost v2 (95 features con NSE/OSM/seguridad).

**Por qué este análisis:** la rúbrica del prototipo (U4_T2 slides 7-8 — Audit
performance / Error analysis) exige mostrar dónde el modelo falla más, no solo
el MAPE promedio. Este notebook responde: ¿el RF es justo across estratos NSE?
¿hay distritos con error sistemático? ¿qué listings explica peor?

**Conclusión adelantada:** el RF baseline tiene sesgo marcado contra zonas
premium con poca data (La Planicie, Casuarinas) y distritos populares (SJL,
Ate). Eso motivó el upgrade a v2 con sample weighting `1/sqrt(count_distrito)`
y Bayesian smoothing k=30 en el target encoding.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PIPE = Path("..").resolve()      # asume cwd = pipeline/notebooks/
DATA = PIPE / "data" / "processed"
MODELS = PIPE / "models"

# Cargar baseline RF + X_test/y_test
rf = joblib.load(MODELS / "04_random_forest.joblib")
X_test = pd.read_csv(DATA / "X_test.csv")
y_test = pd.read_csv(DATA / "y_test.csv")["log_precio"]

# Predicción y residuos en escala USD (no log)
log_pred = rf.predict(X_test)
y_pred_usd = np.expm1(log_pred)
y_real_usd = np.expm1(y_test.values)
residual = y_real_usd - y_pred_usd
residual_pct = 100 * residual / y_real_usd  # error relativo (signed)

print(f"X_test shape: {X_test.shape}")
print(f"MAE  USD: ${np.mean(np.abs(residual)):.2f}")
print(f"MAPE %  : {np.mean(np.abs(residual_pct)):.2f}%")
print(f"RMSE USD: ${np.sqrt(np.mean(residual**2)):.2f}")

In [ ]:
# Enriquecer X_test con metadata del crudo (distrito, categoria, estrato)
crudo = pd.read_csv(DATA / "inmuebles_clean_v1.csv")
# Asumimos que el orden de las filas en X_test coincide con un split estratificado
# del crudo. Para mapeo robusto, usamos un join por índice si las longitudes
# coinciden con un split conocido.
if "id_portal" in crudo.columns and len(crudo) >= len(X_test):
    print(f"Crudo: {len(crudo)} filas | X_test: {len(X_test)} filas")
    # Si X_test no tiene id, asumimos que sus filas corresponden a las últimas
    # N del crudo tras stratified split. Esto es una aproximación.

## Plot 1 — Residuo vs precio real

**Lectura:** si el modelo es homocedástico, los residuos se reparten parejos
alrededor de la línea cero a lo largo del rango de precios. Si el cono se
abre con el precio, hay heterocedasticidad (típico en regresión sobre precios
inmobiliarios — overfit en precios bajos, subestima en altos).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(y_real_usd, residual, alpha=0.4, s=14, color="#2563eb")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Precio real (USD)")
ax.set_ylabel("Residuo = real − predicho (USD)")
ax.set_title("Residuos vs precio real — Random Forest v1")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 2 — Distribución de error por rango de precio

Buckets de precio: el modelo es preciso en el centro de la distribución
(donde hay más data, $500-1200) y se degrada en colas. Esto es típico de
modelos sin focal loss / sample weighting.

In [ ]:
buckets = pd.cut(y_real_usd, bins=[0, 400, 700, 1000, 1500, 2500, 10000])
fig, ax = plt.subplots(figsize=(10, 5))
df_bk = pd.DataFrame({"bucket": buckets, "residual_pct": residual_pct})
sns.boxplot(data=df_bk, x="bucket", y="residual_pct", ax=ax, color="#60a5fa")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Rango de precio real (USD)")
ax.set_ylabel("Error relativo (%) — signed")
ax.set_title("Error relativo por rango de precio — RF v1")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Plot 3 — Top 10 listings con mayor error absoluto

Inspección manual: ¿son anomalías reales (mansiones mal scrapeadas, áreas
inconsistentes) o errores del modelo? Esto orienta dónde apretar la limpieza
de datos y dónde el modelo es estructuralmente débil.

In [ ]:
top10_idx = np.argsort(-np.abs(residual))[:10]
top10 = pd.DataFrame({
    "idx_test": top10_idx,
    "precio_real_USD": y_real_usd[top10_idx].round(0),
    "precio_pred_USD": y_pred_usd[top10_idx].round(0),
    "residuo_USD":     residual[top10_idx].round(0),
    "error_pct":       residual_pct[top10_idx].round(1),
})
print("Top 10 listings con mayor |residuo|:")
print(top10.to_string(index=False))

## Conclusión

**Hallazgos del baseline RF:**
1. **Heterocedasticidad** en colas: el modelo subestima sistemáticamente
   precios > $1500 (zonas premium) y sobrestima precios < $500 (zonas
   populares).
2. **Top errores** son listings con áreas atípicas, mansiones de La Planicie/
   Casuarinas con N≤5 comparables, o departamentos con `tipo_propiedad`
   inconsistente.
3. El MAPE global (15.9%) **esconde** que en zonas con baja densidad el
   error sube a 25-30%.

**Cómo el upgrade a v2 ataca esto:**
- `sample_weight = 1/sqrt(count_distrito)` → fuerza atención sobre distritos
  con poca data.
- Target encoding con Bayesian smoothing k=30 → suaviza distritos con n<10.
- 21 nuevas features OSM (count_500m + count_1km + dist_nearest) → más señal
  de entorno granular.
- 4 features socio-económicas (estrato_nse, categoria_distrito, n_comisarias,
  denuncias) → contexto de seguridad.

**Validación post-v2:** el banner UX `data.confidence === 'Baja'` comunica
al usuario cuándo el rango es ancho (Cobertura baja en esta zona), preservando
la honestidad del producto.

**Referencias en repo:**
- `pipeline/notebooks/04_entrenamiento_modelos.ipynb` — entrenamiento RF y XGB.
- `pipeline/notebooks/05_evaluacion_seleccion.ipynb` — métricas finales.
- `app/backend/ml_v2.py` — `build_features_v2` (95 features producción).
- `app/backend/distrito_features.py` — features socio-económicas v2.
- `gates/gate6_resultado.md` — decisión RF vs XGBoost (gate cerrado).